[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q4/03_wang_evaluation.ipynb)

# R1-Q4 Week 4 — Evaluate cross-dataset transfer to Wang 2023

### This notebook produces `transfer_summary.json`, the headline finding for your Week 4 paper and final presentation.

This notebook applies the classifier you trained in Week 3 to the ComBat-corrected Wang 2023 data, measuring whether a model trained on microarray data can identify cold stress in RNA-seq data. The per-timepoint accuracy results are the headline finding of R1-Q4 and will be the central evidence in your Week 4 paper and final presentation.

By the end of this notebook you will have:

- Cross-dataset prediction accuracy on Wang 2023, broken out by timepoint (0 h, 2 h, 24 h, and 168 h handled per your Week 1 pre-commit), saved as `wang_transfer_accuracy.parquet`.
- A confusion matrix and per-sample predictions showing how the classifier labeled Wang 2023 samples by stress class (`wang_predictions.parquet`).
- A `transfer_summary.json` recording the headline accuracy figure, the per-timepoint breakdown, the comparison against your pre-committed "above-chance" null, and a one-sentence interpretation, ready as the headline finding in your Week 4 paper and final presentation.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R1-Q4 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q4")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Defensive load of seven upstream artifacts and cross-file consistency checks

Notebook 00 wrote `precommit.json`, Notebook 01 wrote `integrated_matrix.parquet` and `integration_summary.json`, Notebook 02 wrote `classifier.pkl`, `classifier_summary.json`, `classifier_metrics.parquet`, and `classifier_confusion.parquet`. All seven sit in `OUTPUT_DIR`. This notebook does no training of its own — it reads what the upstream notebooks produced, applies the classifier to the Wang rows, and writes the transfer-evaluation results. Section 1 is the gatekeeping step.

The work splits across four code cells:

1. **Load the three summary files** — `precommit.json`, `integration_summary.json`, `classifier_summary.json`. Cheap reads that fail fast if something upstream wasn't run.
2. **Refuse on fail** — both upstream gates (`integration_summary` from Notebook 01, `classifier_summary` from Notebook 02) have to clear before this notebook spends time on the heavier file loads. If either reports `"fail"`, Section 1 stops here.
3. **Load the table and pickle files** — `integrated_matrix.parquet`, `classifier.pkl`, `classifier_metrics.parquet`, `classifier_confusion.parquet`. Heavier reads, deferred until the gates have cleared.
4. **Cross-file structural checks** — the seven objects in memory have to agree with each other. The `ood` column in `integrated_matrix` has to match your `wang_168h_handling` decision; the Wang label space has to be a subset of what the classifier was trained on; the classifier bundle has to be the dict-of-two-keys shape Section 2 will unpack.

If any file is missing, malformed, or inconsistent with the others, the load raises and names the notebook to re-run. Re-run that notebook's closeout, then re-run from here.

In [ ]:
import json
import pandas as pd
import joblib

# ----- Paths -----
PRECOMMIT_PATH           = OUTPUT_DIR / "precommit.json"
INTEGRATION_SUMMARY_PATH = OUTPUT_DIR / "integration_summary.json"
CLASSIFIER_SUMMARY_PATH  = OUTPUT_DIR / "classifier_summary.json"
INTEGRATED_PATH          = OUTPUT_DIR / "integrated_matrix.parquet"
CLASSIFIER_PATH          = OUTPUT_DIR / "classifier.pkl"
METRICS_PATH             = OUTPUT_DIR / "classifier_metrics.parquet"
CONFUSION_PATH           = OUTPUT_DIR / "classifier_confusion.parquet"


# ----- precommit.json -----
# Same three-key structure expected by Notebooks 01 and 02. Validates
# that Notebook 00's closeout actually wrote the file we need with the
# decisions in the form downstream code expects.

EXPECTED_PRECOMMIT = {
    "data_source_and_scope": ("source",   {"GEO", "TAIR_NASCArrays"}),
    "wang_168h_handling":    ("decision", {"exclude", "include_as_ood"}),
    "above_chance_null":     ("decision", {"binary_cold_vs_control", "multiclass_behavior"}),
}

if not PRECOMMIT_PATH.exists():
    raise FileNotFoundError(
        f"{PRECOMMIT_PATH} not found. "
        "Run Notebook 00 to its closeout cell, then re-run this cell."
    )

with PRECOMMIT_PATH.open() as f:
    precommit = json.load(f)

for top_key, (field, allowed) in EXPECTED_PRECOMMIT.items():
    if top_key not in precommit:
        raise KeyError(
            f"precommit.json is missing top-level key '{top_key}'. "
            "Re-run Notebook 00's closeout."
        )
    value = precommit[top_key].get(field)
    if value not in allowed:
        raise ValueError(
            f"precommit['{top_key}']['{field}'] = {value!r}, "
            f"but expected one of {sorted(allowed)}. "
            "Fix the relevant Precommit section in Notebook 00 and re-run its closeout."
        )


# ----- integration_summary.json -----
# Notebook 01's Section 8 gate verdict. Flat top-level dict: each part of
# the gate is a top-level key, with "overall" carrying the rollup verdict.
# Access path for the verdict you'll check below: integration_summary["overall"]["verdict"].

ALLOWED_VERDICTS = {"pass", "partial", "fail"}

if not INTEGRATION_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"{INTEGRATION_SUMMARY_PATH} not found. "
        "Run Notebook 01 to its closeout cell, then re-run this cell."
    )

with INTEGRATION_SUMMARY_PATH.open() as f:
    integration_summary = json.load(f)

if "overall" not in integration_summary or "verdict" not in integration_summary["overall"]:
    raise KeyError(
        "integration_summary.json is missing 'overall.verdict'. "
        "Re-run Notebook 01's Section 8 gate cell."
    )

integration_verdict = integration_summary["overall"]["verdict"]
if integration_verdict not in ALLOWED_VERDICTS:
    raise ValueError(
        f"integration_summary['overall']['verdict'] = {integration_verdict!r}, "
        f"but expected one of {sorted(ALLOWED_VERDICTS)}."
    )


# ----- classifier_summary.json -----
# Notebook 02's Section 5 gate verdict, plus run metadata. The gate verdicts
# sit under a "gate" wrapper key (alongside top-level "diagnostics",
# "architecture", "hyperparameters", "split" blocks that document what was
# trained). Access path for the verdict: classifier_summary["gate"]["overall"]["verdict"].
# This is a different shape from integration_summary.json — by design (see
# the next markdown cell).

if not CLASSIFIER_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"{CLASSIFIER_SUMMARY_PATH} not found. "
        "Run Notebook 02 to its closeout cell, then re-run this cell."
    )

with CLASSIFIER_SUMMARY_PATH.open() as f:
    classifier_summary = json.load(f)

if "gate" not in classifier_summary:
    raise KeyError(
        "classifier_summary.json is missing the 'gate' wrapper. "
        "Re-run Notebook 02's Section 5 gate cell."
    )
if "overall" not in classifier_summary["gate"] or "verdict" not in classifier_summary["gate"]["overall"]:
    raise KeyError(
        "classifier_summary.json is missing 'gate.overall.verdict'. "
        "Re-run Notebook 02's Section 5 gate cell."
    )

classifier_verdict = classifier_summary["gate"]["overall"]["verdict"]
if classifier_verdict not in ALLOWED_VERDICTS:
    raise ValueError(
        f"classifier_summary['gate']['overall']['verdict'] = {classifier_verdict!r}, "
        f"but expected one of {sorted(ALLOWED_VERDICTS)}."
    )


# ----- Summary printout -----
print("Loaded three summary files from OUTPUT_DIR.")
print()
print("precommit.json")
for top_key, (field, _) in EXPECTED_PRECOMMIT.items():
    print(f"  {top_key}.{field} = {precommit[top_key][field]!r}")
print()
print("integration_summary.json")
print(f"  overall verdict: {integration_verdict!r}")
print()
print("classifier_summary.json")
print(f"  gate.overall verdict: {classifier_verdict!r}")

**What just loaded:**

- `precommit` — the three-decision dict written by Notebook 00.
- `integration_summary` — the four-part integration gate verdict written by Notebook 01.
- `classifier_summary` — the three-part accuracy gate verdict plus training metadata written by Notebook 02.

**A note on the two summaries' shapes.** The two upstream summary files carry their gate verdict at different paths:

- `integration_summary["overall"]["verdict"]` — flat. The dict's top-level keys are the gate parts.
- `classifier_summary["gate"]["overall"]["verdict"]` — nested under `"gate"`. The wrapper is there because the same file also carries top-level `diagnostics`, `architecture`, `hyperparameters`, and `split` blocks. Without the wrapper, the gate parts would mix with the run-metadata parts in one flat namespace.

Each notebook chose the shape that matched its file's content. The asymmetry isn't an oversight, but it does mean every reader of these files has to look at the schema before indexing into it. The next cell does exactly that — the refusal check on each summary reads with the right path for its own shape.

In [ ]:
# ----- Integration gate -----
if integration_verdict == "fail":
    raise RuntimeError(
        "integration_summary.json reports overall verdict = 'fail'. "
        "Notebook 03 will not run. Return to Notebook 01, fix what failed, "
        "re-run the integration gate, then come back here."
    )

if integration_verdict == "partial":
    integration_rationale = integration_summary["overall"].get("rationale", "(no rationale recorded)")
    print("=" * 60)
    print("WARNING: integration gate verdict was 'partial'.")
    print()
    print("Rationale from Notebook 01:")
    print(f"  {integration_rationale}")
    print()
    print("Notebook 03 will proceed, but make sure you understand")
    print("the trade-off you're carrying forward. Document it in")
    print("your Week 4 paper's limitations paragraph.")
    print("=" * 60)
    print()


# ----- Accuracy gate -----
if classifier_verdict == "fail":
    raise RuntimeError(
        "classifier_summary.json reports gate.overall verdict = 'fail'. "
        "Notebook 03 will not run. Return to Notebook 02, fix what failed "
        "(reconsider C in Section 3, or revisit Notebook 01's integration), "
        "re-run the accuracy gate, then come back here."
    )

if classifier_verdict == "partial":
    classifier_rationale = classifier_summary["gate"]["overall"].get("rationale", "(no rationale recorded)")
    print("=" * 60)
    print("WARNING: accuracy gate verdict was 'partial'.")
    print()
    print("Rationale from Notebook 02:")
    print(f"  {classifier_rationale}")
    print()
    print("Notebook 03 will proceed, but make sure you understand")
    print("the trade-off you're carrying forward. Document it in")
    print("your Week 4 paper's limitations paragraph.")
    print("=" * 60)
    print()


if integration_verdict == "pass" and classifier_verdict == "pass":
    print("Both upstream gates cleared: integration='pass', accuracy='pass'.")
elif integration_verdict != "fail" and classifier_verdict != "fail":
    print(
        f"Upstream gates cleared with caveats: "
        f"integration={integration_verdict!r}, accuracy={classifier_verdict!r}. "
        "See warnings above."
    )

In [ ]:
# ----- integrated_matrix.parquet -----
# Wide table from Notebook 01's Section 9. Samples are rows; columns are
# gene values on the ComBat-corrected scale, plus three metadata columns:
# batch, stress_label, ood.

if not INTEGRATED_PATH.exists():
    raise FileNotFoundError(
        f"{INTEGRATED_PATH} not found. "
        "Run Notebook 01 to its closeout cell, then re-run this cell."
    )

integrated_matrix = pd.read_parquet(INTEGRATED_PATH)

REQUIRED_META_COLS = {"batch", "stress_label", "ood"}
missing_meta = REQUIRED_META_COLS - set(integrated_matrix.columns)
if missing_meta:
    raise KeyError(
        f"integrated_matrix.parquet is missing metadata column(s): {sorted(missing_meta)}. "
        "Re-run Notebook 01 — Section 9 assembles these columns."
    )


# ----- classifier.pkl -----
# joblib-pickled dict bundling the StandardScaler and the trained
# LogisticRegression. Section 2 unpacks this and applies both to the
# Wang rows in turn.

if not CLASSIFIER_PATH.exists():
    raise FileNotFoundError(
        f"{CLASSIFIER_PATH} not found. "
        "Run Notebook 02 to its closeout cell, then re-run this cell."
    )

classifier_bundle = joblib.load(CLASSIFIER_PATH)

if not isinstance(classifier_bundle, dict):
    raise TypeError(
        f"classifier.pkl loaded as {type(classifier_bundle).__name__}, expected dict. "
        "Re-run Notebook 02 — Section 6 should save a dict with keys 'scaler' and 'classifier'."
    )

REQUIRED_BUNDLE_KEYS = {"scaler", "classifier"}
missing_bundle = REQUIRED_BUNDLE_KEYS - set(classifier_bundle.keys())
if missing_bundle:
    raise KeyError(
        f"classifier.pkl is missing bundle key(s): {sorted(missing_bundle)}. "
        f"Found keys: {sorted(classifier_bundle.keys())}. "
        "Re-run Notebook 02's Section 6 — the bundle should be {'scaler': ..., 'classifier': ...}."
    )


# ----- classifier_metrics.parquet -----
# Per-class precision/recall/F1/support table from Notebook 02 Section 4.
# Includes macro_avg and weighted_avg rows alongside the per-class rows.
# Classes live in a 'class' column, not in the index.

if not METRICS_PATH.exists():
    raise FileNotFoundError(
        f"{METRICS_PATH} not found. "
        "Run Notebook 02 to its closeout cell, then re-run this cell."
    )

classifier_metrics = pd.read_parquet(METRICS_PATH)

REQUIRED_METRICS_COLS = {"class", "precision", "recall", "f1", "support"}
missing_metrics_cols = REQUIRED_METRICS_COLS - set(classifier_metrics.columns)
if missing_metrics_cols:
    raise KeyError(
        f"classifier_metrics.parquet is missing column(s): {sorted(missing_metrics_cols)}. "
        "Re-run Notebook 02 — Section 4 assembles these columns."
    )


# ----- classifier_confusion.parquet -----
# Square confusion matrix from Notebook 02 Section 4. Index name 'true',
# columns name 'predicted'. Not used in computation by Notebook 03 (the
# label-space mismatch makes a confusion-vs-confusion comparison undefined
# for most NB02 classes — see Section 5 prose), but loaded here so the
# full defensive-load contract lives in one place.

if not CONFUSION_PATH.exists():
    raise FileNotFoundError(
        f"{CONFUSION_PATH} not found. "
        "Run Notebook 02 to its closeout cell, then re-run this cell."
    )

classifier_confusion = pd.read_parquet(CONFUSION_PATH)


# ----- Summary printout -----
print("Loaded four table and pickle files from OUTPUT_DIR.")
print()
print("integrated_matrix.parquet")
print(f"  shape:           {integrated_matrix.shape[0]:,} samples x {integrated_matrix.shape[1]:,} columns")
print(f"  batch breakdown: {integrated_matrix['batch'].value_counts().to_dict()}")
print()
print("classifier.pkl")
print(f"  scaler:     {type(classifier_bundle['scaler']).__name__}")
print(f"  classifier: {type(classifier_bundle['classifier']).__name__}")
print(f"  classes:    {list(classifier_bundle['classifier'].classes_)}")
print()
print("classifier_metrics.parquet")
print(f"  rows: {len(classifier_metrics)} (per-class + macro_avg + weighted_avg)")
print()
print("classifier_confusion.parquet")
print(f"  shape: {classifier_confusion.shape[0]} x {classifier_confusion.shape[1]}")

In [ ]:
# ----- integrated_matrix structural integrity -----

if not integrated_matrix.index.is_unique:
    n_dupes = integrated_matrix.index.duplicated().sum()
    raise ValueError(
        f"integrated_matrix has {n_dupes:,} duplicate sample ID(s). "
        "Re-run Notebook 01 — Section 9's concat should not produce duplicates."
    )

if integrated_matrix["ood"].dtype != bool:
    raise TypeError(
        f"integrated_matrix['ood'] dtype is {integrated_matrix['ood'].dtype}, expected bool. "
        "Re-run Notebook 01's Section 9."
    )

gene_cols = integrated_matrix.columns.difference(["batch", "stress_label", "ood"])
n_nans = integrated_matrix[gene_cols].isna().sum().sum()
if n_nans > 0:
    raise ValueError(
        f"integrated_matrix gene columns contain {n_nans:,} NaN value(s). "
        "Re-run Notebook 01 — Section 5's gene intersection should leave no missing values."
    )

unexpected_batches = set(integrated_matrix["batch"].unique()) - {"atgenexpress", "wang"}
if unexpected_batches:
    raise ValueError(
        f"integrated_matrix['batch'] contains unexpected values: {sorted(unexpected_batches)}. "
        "Re-run Notebook 01 — Section 9 assigns batch labels."
    )


# ----- precommit <-> integrated_matrix consistency -----
# The wang_168h_handling decision shapes the ood column. Drift between
# the two would mean Notebook 01 ran on a different precommit than the
# one currently on disk.

handling   = precommit["wang_168h_handling"]["decision"]
wang_mask  = integrated_matrix["batch"] == "wang"
atgen_mask = integrated_matrix["batch"] == "atgenexpress"
n_wang_ood  = int(integrated_matrix.loc[wang_mask,  "ood"].sum())
n_atgen_ood = int(integrated_matrix.loc[atgen_mask, "ood"].sum())

if n_atgen_ood > 0:
    raise ValueError(
        f"AtGenExpress rows should never carry ood=True, but {n_atgen_ood} do. "
        "Re-run Notebook 01 — Section 9 assigns False to all AtGenExpress rows."
    )

if handling == "exclude" and n_wang_ood > 0:
    raise ValueError(
        f"precommit.wang_168h_handling = 'exclude', but {n_wang_ood} Wang row(s) "
        "carry ood=True. The two files disagree — most likely Notebook 01 ran on a "
        "different precommit and was never re-run after the decision changed. "
        "Re-run Notebook 01 from the top."
    )

if handling == "include_as_ood" and n_wang_ood == 0:
    raise ValueError(
        "precommit.wang_168h_handling = 'include_as_ood', but no Wang rows carry "
        "ood=True. The two files disagree — re-run Notebook 01 from the top."
    )


# ----- Wang label space <-> classifier label space -----
# Wang rows carry exactly two stress_label values: "control" and "cold_stress".
# Both must be in the classifier's training class space — otherwise the
# transfer evaluation has no meaningful per-class recall to report.

wang_labels = set(integrated_matrix.loc[wang_mask, "stress_label"].unique())
EXPECTED_WANG_LABELS = {"control", "cold_stress"}
if wang_labels != EXPECTED_WANG_LABELS:
    raise ValueError(
        f"Wang rows carry stress_label values {sorted(wang_labels)}, "
        f"but expected exactly {sorted(EXPECTED_WANG_LABELS)}. "
        "Re-run Notebook 01 — Section 6's harmonize_stress_label should map Wang's "
        "treatment timepoints to {'control', 'cold_stress'} and nothing else."
    )

classifier_classes = set(classifier_bundle["classifier"].classes_)
missing_wang_classes = EXPECTED_WANG_LABELS - classifier_classes
if missing_wang_classes:
    raise ValueError(
        f"The classifier was not trained on class(es) {sorted(missing_wang_classes)}. "
        f"Classifier classes: {sorted(classifier_classes)}. "
        "The transfer evaluation can't compare against a class the classifier doesn't know. "
        "Re-run Notebook 02 — Section 2's train/test split should preserve all AtGenExpress "
        "stress_label values in the training set."
    )


# ----- classifier_metrics <-> classifier label space -----
# classifier_metrics.parquet has classes in a 'class' column (not the index)
# alongside two average rows. The per-class subset should match the
# classifier's class space exactly; if it doesn't, the metrics file was
# generated against a different classifier than the one in the bundle.

AVG_ROW_LABELS = {"macro_avg", "weighted_avg"}
metrics_per_class = set(classifier_metrics["class"]) - AVG_ROW_LABELS
if metrics_per_class != classifier_classes:
    raise ValueError(
        f"classifier_metrics per-class rows {sorted(metrics_per_class)} do not match "
        f"classifier_bundle class space {sorted(classifier_classes)}. "
        "Re-run Notebook 02 — the metrics file and the pickled classifier should "
        "describe the same model."
    )


# ----- classifier_confusion shape -----
# Square matrix, rows and columns share the same class space.

if classifier_confusion.shape[0] != classifier_confusion.shape[1]:
    raise ValueError(
        f"classifier_confusion.parquet is not square: shape {classifier_confusion.shape}. "
        "Re-run Notebook 02 — Section 4 builds the matrix with classifier.classes_ on both axes."
    )

if set(classifier_confusion.index) != classifier_classes:
    raise ValueError(
        f"classifier_confusion row labels {sorted(set(classifier_confusion.index))} "
        f"do not match classifier class space {sorted(classifier_classes)}. "
        "Re-run Notebook 02 — Section 4."
    )

if set(classifier_confusion.columns) != classifier_classes:
    raise ValueError(
        f"classifier_confusion column labels {sorted(set(classifier_confusion.columns))} "
        f"do not match classifier class space {sorted(classifier_classes)}. "
        "Re-run Notebook 02 — Section 4."
    )


print("All cross-file consistency checks passed.")
print()
print(f"Wang sample counts: in-distribution = {(wang_mask & ~integrated_matrix['ood']).sum()}, "
      f"out-of-distribution = {n_wang_ood}")
print(f"Classifier class space ({len(classifier_classes)} classes): {sorted(classifier_classes)}")

**What's now in memory:**

- `precommit` — Notebook 00's three-decision dict. Section 2 reads `wang_168h_handling`; Section 4 branches on `above_chance_null`.
- `integration_summary` — Notebook 01's integration gate. Surfaced in the Week 4 paper's methods section as evidence that the cross-platform integration step was inspected before training.
- `classifier_summary` — Notebook 02's accuracy gate plus training metadata. Section 5's comparison artifact reads from `classifier_summary["diagnostics"]`; the gate verdict feeds the paper's limitations paragraph.
- `integrated_matrix` — the full integrated dataset. Wang rows (`batch == "wang"`) are the transfer test set for Section 2.
- `classifier_bundle` — `{"scaler": ..., "classifier": ...}`. Section 2's practice cell unpacks this and applies both in turn to the Wang rows.
- `classifier_metrics` — per-class precision/recall/F1/support, with `macro_avg` and `weighted_avg` rows that Section 5 filters out before joining on `class`.
- `classifier_confusion` — within-platform confusion matrix. Read for completeness; not used in any computation (the label-space mismatch makes a direct confusion-vs-confusion comparison undefined for most NB02 classes — see Section 5 prose for why).

All seven objects have been checked structurally and against each other. Section 2 takes the integrated matrix, filters to Wang rows, applies the classifier bundle, and partitions the predictions by the `ood` flag.

### 2) Extract Wang rows, apply classifier, partition by OOD

The classifier bundle in `classifier_bundle` was trained on the AtGenExpress portion of `integrated_matrix`. This section applies it to the Wang portion — the cross-platform transfer test the R1-Q4 question is built around. Three operations:

1. **Extract.** Filter `integrated_matrix` to rows where `batch == "wang"`.
2. **Apply.** Run the scaler and the classifier on the gene columns of those rows; collect predicted labels and per-sample maximum class probabilities.
3. **Partition.** Split the results by the `ood` flag. In-distribution Wang samples feed Sections 3–6; out-of-distribution Wang samples (if your precommit was `include_as_ood`) get evaluated in parallel by Section 3 only, never folded into the gate verdict.

You write the function that does all three. The cell after the practice cell calls it, assembles a DataFrame from the partitioned output, and writes `transfer_predictions.parquet` to `OUTPUT_DIR`.

**What to produce.** A function `apply_classifier_to_wang(bundle, integrated_matrix)` returning:

```python
{
    "in_distribution": {
        "sample_id":         array,
        "predicted_label":   array,
        "prediction_prob":   array,
        "true_stress_label": array,
    },
    "out_of_distribution": {
        # same four-key shape; arrays may be empty on the 'exclude' path
    },
}
```

All four arrays within a partition are the same length (one entry per sample in that partition). `prediction_prob` is the maximum class probability for each prediction — the classifier's confidence in the label it assigned, used by Section 4 as a diagnostic.

The two-partition return is deliberate. Downstream sections evaluate in-distribution and out-of-distribution separately (see the rationale in the scoping handoff and in the R1-Q4 question page); pre-partitioning here means each downstream section just indexes into the partition it needs rather than re-applying the `ood` filter every time.

In [ ]:
# Practice 2.1 — your code here.
#
# Goal: implement apply_classifier_to_wang. Apply the trained {scaler,
# classifier} bundle to the Wang rows of integrated_matrix and return
# predictions partitioned by the ood flag.

def apply_classifier_to_wang(bundle, integrated_matrix):
    """
    Apply a trained {scaler, classifier} bundle to the Wang rows of
    integrated_matrix, partitioned by the ood flag.

    Parameters
    ----------
    bundle : dict
        From Notebook 02's classifier.pkl. Keys are 'scaler' (a fitted
        StandardScaler) and 'classifier' (a fitted LogisticRegression).
    integrated_matrix : pandas.DataFrame
        From Notebook 01's integrated_matrix.parquet. Rows are samples
        indexed by sample_id; columns are gene values plus three
        metadata columns ('batch', 'stress_label', 'ood').

    Returns
    -------
    dict
        Two keys, 'in_distribution' and 'out_of_distribution'. Each
        value is a dict with four 1D arrays of equal length:
        sample_id, predicted_label, prediction_prob, true_stress_label.
        prediction_prob is the row-wise max of predict_proba — the
        classifier's confidence in the predicted class. On the
        'exclude' precommit path the out_of_distribution arrays are
        length zero, which is expected.
    """
    # Step 1: Unpack the bundle. It is a dict with two keys, 'scaler'
    #         and 'classifier'.
    #
    # Step 2: Filter integrated_matrix to Wang rows. The condition is
    #         batch == "wang". The filtered DataFrame's index carries
    #         the sample IDs you will need for the return dict.
    #
    # Step 3: Identify the gene columns. integrated_matrix has three
    #         metadata columns ('batch', 'stress_label', 'ood') and the
    #         rest are gene values. The scaler was fit on the gene
    #         columns in Notebook 02, so the column set has to match
    #         when you call .transform() now.
    #         Hint: pandas.Index.difference does this in one line.
    #
    # Step 4: Apply the scaler. scaler.transform(X) returns a 2D array
    #         with the same row count as X.
    #
    # Step 5: Apply the classifier. You need both calls:
    #             classifier.predict(X)         -> 1D array of labels
    #             classifier.predict_proba(X)   -> 2D array, one row
    #                                              per sample, one column
    #                                              per class.
    #         The prediction_prob field in the return dict is the
    #         row-wise max of predict_proba — see numpy.ndarray.max
    #         with the axis= argument.
    #
    # Step 6: Partition by the ood flag. Rows where ood == False go
    #         to "in_distribution"; rows where ood == True go to
    #         "out_of_distribution". A single boolean mask handles
    #         both paths uniformly — do not special-case the
    #         'exclude' precommit path; the OOD arrays will just come
    #         out length zero.
    #
    # Step 7: Build and return the dict. Each partition's value is a
    #         dict with the four keys named in the docstring. The
    #         true_stress_label values come from the stress_label
    #         column of the filtered Wang rows.

    raise NotImplementedError(
        "Practice 2.1 — implement the function body using the seven "
        "steps in the scaffold above."
    )


# Sanity prints — uncomment after the function body is written, to
# spot-check the return shape before the next cell consumes it.
# preds = apply_classifier_to_wang(classifier_bundle, integrated_matrix)
# in_part  = preds["in_distribution"]
# ood_part = preds["out_of_distribution"]
# print(f"in_distribution:      {len(in_part['sample_id'])} samples")
# print(f"out_of_distribution:  {len(ood_part['sample_id'])} samples")
# print(f"predicted labels (in-dist): {sorted(set(in_part['predicted_label']))}")

In [ ]:
# ----- Call the function -----
predictions = apply_classifier_to_wang(classifier_bundle, integrated_matrix)


# ----- Validate the function's return shape -----
# If the student's function returned the wrong shape, the DataFrame
# assembly below would fail with a confusing pandas/numpy error. These
# explicit checks point at the actual problem.

EXPECTED_PARTITIONS = {"in_distribution", "out_of_distribution"}
EXPECTED_ARRAYS = ["sample_id", "predicted_label", "prediction_prob", "true_stress_label"]

if not isinstance(predictions, dict):
    raise TypeError(
        f"apply_classifier_to_wang returned {type(predictions).__name__}, "
        "but a dict is expected. See the docstring's Returns block."
    )

missing_partitions = EXPECTED_PARTITIONS - set(predictions.keys())
if missing_partitions:
    raise KeyError(
        f"apply_classifier_to_wang's return is missing partition key(s): "
        f"{sorted(missing_partitions)}. Expected exactly {sorted(EXPECTED_PARTITIONS)}."
    )

for partition_name, partition in predictions.items():
    if not isinstance(partition, dict):
        raise TypeError(
            f"predictions[{partition_name!r}] is {type(partition).__name__}, "
            "but a dict is expected."
        )
    missing_arrays = set(EXPECTED_ARRAYS) - set(partition.keys())
    if missing_arrays:
        raise KeyError(
            f"predictions[{partition_name!r}] is missing array(s): {sorted(missing_arrays)}. "
            f"Expected exactly {sorted(EXPECTED_ARRAYS)}."
        )
    lengths = {key: len(partition[key]) for key in EXPECTED_ARRAYS}
    if len(set(lengths.values())) > 1:
        raise ValueError(
            f"predictions[{partition_name!r}] arrays have mismatched lengths: {lengths}. "
            "All four arrays in a partition must be the same length."
        )


# ----- Assemble the predictions DataFrame -----
# One row per Wang sample; an 'ood' column records which partition the
# row came from. The two partitions are concatenated; sample_id becomes
# the index (matching integrated_matrix's indexing convention).

def partition_to_frame(partition, ood_flag):
    frame = pd.DataFrame({key: partition[key] for key in EXPECTED_ARRAYS})
    frame["ood"] = ood_flag
    return frame

in_df  = partition_to_frame(predictions["in_distribution"],     ood_flag=False)
ood_df = partition_to_frame(predictions["out_of_distribution"], ood_flag=True)

transfer_predictions = pd.concat([in_df, ood_df], ignore_index=True).set_index("sample_id")


# ----- Row-count cross-check -----
# Total predictions should equal total Wang rows in integrated_matrix.
# A mismatch means the function dropped or duplicated samples in
# Steps 2 or 6.

n_wang = int((integrated_matrix["batch"] == "wang").sum())
if len(transfer_predictions) != n_wang:
    raise ValueError(
        f"transfer_predictions has {len(transfer_predictions)} rows but "
        f"integrated_matrix has {n_wang} Wang rows. The function's filter "
        "or partition step dropped or duplicated samples."
    )


# ----- Write to disk -----
transfer_predictions_path = OUTPUT_DIR / "transfer_predictions.parquet"
transfer_predictions.to_parquet(transfer_predictions_path)


# ----- Sanity-load -----
# Confirm the write round-tripped cleanly. parquet round-trips are
# almost always fine, but the check costs nothing and surfaces the
# rare drive-mount glitch before Section 3 tries to read the file.
reloaded = pd.read_parquet(transfer_predictions_path)
if len(reloaded) != len(transfer_predictions):
    raise IOError(
        f"transfer_predictions.parquet round-trip failed: wrote "
        f"{len(transfer_predictions)} rows, read back {len(reloaded)}."
    )


# ----- Report -----
print(f"transfer_predictions.parquet written to {transfer_predictions_path}")
print(f"  shape: {len(transfer_predictions)} samples x {transfer_predictions.shape[1]} columns")
print(f"  partition: in-distribution = {(~transfer_predictions['ood']).sum()}, "
      f"out-of-distribution = {transfer_predictions['ood'].sum()}")
print()
print("Preview (first 8 rows):")
print(transfer_predictions.head(8).to_string())

**What just happened:**

- Your `apply_classifier_to_wang` function unpacked the bundle, filtered to Wang rows, applied the scaler and classifier, and split predictions by the `ood` flag.
- The two partitions were concatenated into `transfer_predictions` and written to `transfer_predictions.parquet`. Each row carries the sample ID (as index), the predicted label, the maximum class probability for that prediction, the true stress label from `integrated_matrix`, and the `ood` flag.

**In memory now:**

- `predictions` — the raw dict returned by your function. Sections 3 and 4 read this directly; reading from memory is faster than re-reading the parquet for downstream sections in the same notebook run, and skipping the file round-trip avoids any chance of a parquet schema quirk affecting downstream computation.
- `transfer_predictions` — the assembled DataFrame, also on disk. This is the artifact your Week 4 paper cites as the "predictions table" — every Wang sample, what the classifier called it, how confident it was.

**Forward pointer.** Section 3 reads `predictions` (not the parquet) and computes per-class precision, recall, F1, and the confusion matrix — separately for the in-distribution Wang samples and (on the `include_as_ood` path) for the out-of-distribution ones. The parallel evaluation is what produces the two parquet files Section 3 writes: `transfer_metrics.parquet` (in-distribution only, used by the gate) and a side report of OOD metrics that ends up in `transfer_summary.json` rather than its own parquet.

### 3) Per-class metrics and confusion matrix, in-distribution and OOD

Section 2 produced `predictions` — a dict with two partitions, in-distribution and out-of-distribution, each carrying predicted labels, prediction probabilities, and true labels for the Wang samples it covers. Section 3 turns those predictions into the standard transfer-evaluation outputs: per-class metrics and a confusion matrix.

The work runs in parallel across the two partitions, with three deliberate design choices that match the R1-Q4 scoping decisions:

1. **In-distribution and out-of-distribution are evaluated separately, not together.** The two partitions answer different questions: in-distribution asks whether the classifier recognises cold stress at timepoints comparable to its AtGenExpress training data; out-of-distribution asks whether it still recognises cold stress at 168 h, an order of magnitude further out than any training timepoint. Conflating them would produce a single number that answers neither cleanly.

2. **The confusion matrix is rectangular, not square.** Wang carries exactly two true labels (`control` and `cold_stress`); the classifier was trained on roughly eight (depending on your data-source precommit). A square 8×8 matrix would have 6 rows entirely zero — no Wang sample carries those true labels. A rectangular matrix (2 true rows × N predicted columns) shows the interpretable question directly: *for actual Wang control samples, where did the classifier put them? For actual Wang cold_stress samples, where?*

3. **Per-class and average metrics are computed over Wang's label space only.** sklearn's `precision_recall_fscore_support` accepts a `labels=` argument that pins the evaluation to a specific class set. Pinning it to `["control", "cold_stress"]` means the macro and weighted averages are over the two labels Wang actually has, not over whatever class set appears in the predictions. This makes Section 3's per-class numbers directly comparable to Notebook 02's corresponding rows in `classifier_metrics.parquet` — the comparison artifact in Section 5 depends on this alignment.

Two parquet files get written: `transfer_metrics.parquet` (in-distribution per-class metrics) and `transfer_confusion.parquet` (in-distribution rectangular confusion). The out-of-distribution counterparts stay in memory — Section 6 includes them as side fields in `transfer_summary.json` rather than giving them their own parquet, because the OOD numbers don't enter the gate verdict and don't get joined against any other artifact downstream.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

WANG_LABELS = ["control", "cold_stress"]


def compute_metrics_and_confusion(true_labels, predicted_labels, classifier_classes):
    """
    Compute per-class metrics and a rectangular confusion matrix for one
    Wang partition (in-distribution or out-of-distribution).

    Metrics are computed against the Wang label space, regardless of what
    classes actually appear in predicted_labels. This means the result
    DataFrame has a stable shape across both partitions and on either
    precommit path.

    Parameters
    ----------
    true_labels : array-like
        Ground-truth labels for this partition (from integrated_matrix's
        stress_label column).
    predicted_labels : array-like
        Labels returned by the classifier for the same samples.
    classifier_classes : array-like
        The full label space the classifier was trained on (i.e.,
        classifier.classes_). Determines the columns of the confusion matrix.

    Returns
    -------
    metrics_df : pandas.DataFrame
        Columns: ['class', 'precision', 'recall', 'f1', 'support'].
        Rows: WANG_LABELS in fixed order, followed by 'macro_avg' and
        'weighted_avg' rows. Same column shape as Notebook 02's
        classifier_metrics.parquet, so Section 5 can join on 'class'.
    confusion_df : pandas.DataFrame
        Rectangular: len(WANG_LABELS) rows × len(classifier_classes)
        columns. Index name 'true'; columns name 'predicted'. Stable
        shape regardless of which classes actually got predicted.
    """
    # ----- Per-class metrics, pinned to Wang labels -----
    precision, recall, f1, support = precision_recall_fscore_support(
        true_labels, predicted_labels,
        labels=WANG_LABELS,
        zero_division=0,
    )
    per_class_rows = pd.DataFrame({
        "class":     WANG_LABELS,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "support":   support,
    })

    # ----- Macro and weighted averages, also pinned to Wang labels -----
    # Averaging over the same labels as the per-class rows means the
    # averages summarise the same evaluation, not a wider one that
    # would also include classes Wang doesn't carry.
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        true_labels, predicted_labels,
        labels=WANG_LABELS,
        average="macro",
        zero_division=0,
    )
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
        true_labels, predicted_labels,
        labels=WANG_LABELS,
        average="weighted",
        zero_division=0,
    )
    total_support = int(support.sum())
    avg_rows = pd.DataFrame({
        "class":     ["macro_avg",  "weighted_avg"],
        "precision": [prec_macro,   prec_w],
        "recall":    [rec_macro,    rec_w],
        "f1":        [f1_macro,     f1_w],
        "support":   [total_support, total_support],
    })

    metrics_df = pd.concat([per_class_rows, avg_rows], ignore_index=True)

    # ----- Rectangular confusion -----
    # Rows = Wang labels (in fixed order); columns = classifier classes.
    # pd.crosstab returns a sparse table — only the (true, predicted)
    # combinations that actually appeared. reindex forces the full shape,
    # filling absent combinations with zero counts. This keeps the
    # confusion matrix shape stable across partitions and precommit paths.
    cm = pd.crosstab(
        pd.Series(true_labels, name="true"),
        pd.Series(predicted_labels, name="predicted"),
    )
    confusion_df = cm.reindex(
        index=WANG_LABELS,
        columns=list(classifier_classes),
        fill_value=0,
    )
    confusion_df.index.name = "true"
    confusion_df.columns.name = "predicted"

    return metrics_df, confusion_df


print("compute_metrics_and_confusion is defined.")
print(f"  Evaluation labels (rows of confusion): {WANG_LABELS}")
print(f"  Confusion column shape comes from classifier.classes_ at call time.")

In [ ]:
# Pull in-distribution arrays from predictions.
in_part = predictions["in_distribution"]

# Compute metrics and confusion.
transfer_metrics_in, transfer_confusion_in = compute_metrics_and_confusion(
    true_labels=in_part["true_stress_label"],
    predicted_labels=in_part["predicted_label"],
    classifier_classes=classifier_bundle["classifier"].classes_,
)


# ----- Display -----
n_in = len(in_part["sample_id"])
print(f"In-distribution Wang samples: {n_in}")
print()
print("Per-class metrics:")
print(transfer_metrics_in.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print()
print("Confusion matrix (rows = true Wang labels, columns = classifier classes):")
print(transfer_confusion_in.to_string())


# ----- Write to disk -----
metrics_path   = OUTPUT_DIR / "transfer_metrics.parquet"
confusion_path = OUTPUT_DIR / "transfer_confusion.parquet"

transfer_metrics_in.to_parquet(metrics_path)
transfer_confusion_in.to_parquet(confusion_path)

# Sanity-load round-trip on both files.
metrics_reloaded   = pd.read_parquet(metrics_path)
confusion_reloaded = pd.read_parquet(confusion_path)
if len(metrics_reloaded) != len(transfer_metrics_in):
    raise IOError(f"transfer_metrics.parquet round-trip failed: row count mismatch.")
if confusion_reloaded.shape != transfer_confusion_in.shape:
    raise IOError(f"transfer_confusion.parquet round-trip failed: shape mismatch.")

print()
print(f"Written: {metrics_path}")
print(f"Written: {confusion_path}")

In [ ]:
ood_part = predictions["out_of_distribution"]
n_ood = len(ood_part["sample_id"])

if n_ood == 0:
    # Empty OOD partition. Either:
    # - precommit["wang_168h_handling"]["decision"] == "exclude", so 168 h
    #   samples never entered the integrated matrix; or
    # - the include_as_ood path was taken but Wang's run somehow had no
    #   168 h samples (extremely unlikely; flagged by Section 1's cross-check).
    transfer_metrics_ood   = None
    transfer_confusion_ood = None

    print("No out-of-distribution samples to evaluate.")
    print()
    print(f"  precommit.wang_168h_handling: "
          f"{precommit['wang_168h_handling']['decision']!r}")
    print()
    print("Section 6 will record OOD as not-applicable in transfer_summary.json.")

else:
    transfer_metrics_ood, transfer_confusion_ood = compute_metrics_and_confusion(
        true_labels=ood_part["true_stress_label"],
        predicted_labels=ood_part["predicted_label"],
        classifier_classes=classifier_bundle["classifier"].classes_,
    )

    print(f"Out-of-distribution Wang samples: {n_ood}")
    print()
    print("Per-class metrics (OOD):")
    print(transfer_metrics_ood.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    print()
    print("Confusion matrix (OOD):")
    print(transfer_confusion_ood.to_string())
    print()
    print("These OOD numbers do not enter the gate verdict (Section 6).")
    print("They flow into transfer_summary.json as side fields for the paper's discussion.")
    print()
    print("Note on Wang's OOD label space: the 168 h timepoint is cold-stress only,")
    print("so the 'control' row of the per-class metrics will have support=0 and")
    print("precision/recall both 0. This is structural, not a classifier failure —")
    print("there are no actual control samples at 168 h to evaluate against.")

**What just happened:**

- Per-class metrics and a confusion matrix were computed for each Wang partition. The in-distribution results were written to `transfer_metrics.parquet` and `transfer_confusion.parquet`; the OOD results stay in memory.
- All four DataFrames share a stable shape across partitions and precommit paths. The `transfer_metrics_*` DataFrames always have exactly four rows in the same order — `control`, `cold_stress`, `macro_avg`, `weighted_avg`. The `transfer_confusion_*` DataFrames are always rectangular, two rows by N columns (where N is the size of the classifier's class space). Stable shapes mean Sections 4–6 don't have to defensively handle absent rows or columns.

**A note on the macro and weighted averages.** Both are computed over Wang's two labels only, not over the classifier's full ~8-class space. This is the right thing to do for *Wang-side* evaluation (the question is "how does the classifier do on Wang's labels?"), but it means the average rows in `transfer_metrics.parquet` are not directly comparable to the average rows in `classifier_metrics.parquet` — those were computed over Notebook 02's full class space. Cross-notebook comparison happens at the per-class level only; Section 5's comparison artifact joins on the `control` and `cold_stress` rows specifically and ignores the averages.

**In memory now:**

- `transfer_metrics_in`, `transfer_confusion_in` — also on disk.
- `transfer_metrics_ood`, `transfer_confusion_ood` — `None` on the `exclude` precommit path; non-None DataFrames on the `include_as_ood` path with at least one 168 h sample.
- `WANG_LABELS` — the two-element list `["control", "cold_stress"]`. Section 5 references it when filtering `classifier_metrics` for the per-class join.
- `compute_metrics_and_confusion` — the helper function, still callable if you want to explore (e.g., recomputing on a subset of in-distribution samples).

**Forward pointer.** Section 4 runs the null comparison. It reads the in-distribution `predictions["in_distribution"]` directly (not the parquets) and branches on `precommit["above_chance_null"]["decision"]` — balanced accuracy on the binary path, per-class recall on the multiclass path. The null comparison is the second of NB03's two practice cells.

### 4) Null comparison: is transfer accuracy above chance?

You committed in Notebook 00 to one of two ways to measure whether the classifier's transfer performance on Wang is above what chance would produce. Section 4 runs that measurement.

**The two paths.** Your precommit lives at `precommit["above_chance_null"]["decision"]`:

- `"binary_cold_vs_control"` — collapse the classifier's multi-class predictions to a yes/no decision ("is this sample cold-stressed or not?"), score with balanced accuracy, and compare against a 0.50 null. Balanced accuracy is preferred over raw accuracy on Wang because the dataset is imbalanced (3 controls and 9 cold samples per the Wang Design Finding Memo) and raw accuracy on a 25/75 split has a deceptively high chance baseline.
- `"multiclass_behavior"` — keep the full multi-class prediction distribution and ask whether each Wang label gets called correctly at a rate above the uniform-chance null of 1/N (where N is the classifier's class count: 8 or 9 depending on your `data_source_and_scope` precommit). Each Wang label gets its own point estimate and CI.

**Bootstrap confidence intervals.** Both branches run a stratified bootstrap to produce a 95% CI alongside the point estimate. The bootstrap is *stratified* — resamples are drawn within each true-label class separately, preserving the per-class sample sizes across iterations. This matters because a naive (non-stratified) bootstrap on a 3-control / 9-cold dataset could occasionally produce a resample with zero controls or zero colds, on which neither balanced accuracy nor "control" recall is well-defined. Stratifying makes every iteration a meaningful evaluation.

The bootstrap runs `n_bootstrap = 1000` iterations with `random_state = 0`, fixed for reproducibility — re-running this section on the same predictions array produces identical CI bounds.

**What you write.** A function `compute_null_comparison(predicted_labels, true_labels, precommit, classifier_classes)` that branches on `precommit["above_chance_null"]["decision"]` and returns a result dict. The dict's value shapes depend on the branch:

```python
# binary path
{
    "metric_name":     "balanced_accuracy",
    "null_value":      0.50,
    "point_estimate":  scalar,
    "ci_lower":        scalar,
    "ci_upper":        scalar,
}

# multiclass path
{
    "metric_name":     "per_class_recall",
    "null_value":      1.0 / N,
    "point_estimate":  {"control": scalar, "cold_stress": scalar},
    "ci_lower":        {"control": scalar, "cold_stress": scalar},
    "ci_upper":        {"control": scalar, "cold_stress": scalar},
}
```

The cell below the practice defines a `stratified_bootstrap` helper for you to call — you don't have to write the bootstrap loop from scratch. The cell after the practice calls your function, validates its return shape, and displays the point estimate alongside the CI and the null value. Section 6 (the gate) reads the result dict and folds it into `transfer_summary.json`.

In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score, recall_score


def stratified_bootstrap(true_arr, pred_arr, metric_fn,
                          n_bootstrap=1000, random_state=0):
    """
    Run a stratified bootstrap of a classification metric.

    Resamples are drawn within each true-label class separately, so the
    per-class sample sizes are preserved across iterations. This matters
    for small samples where a naive (non-stratified) bootstrap could
    accidentally produce a resample with all-one-class.

    Parameters
    ----------
    true_arr : array-like
        Ground-truth labels.
    pred_arr : array-like, same length as true_arr
        Predicted labels for the same samples.
    metric_fn : callable
        Takes (true, pred) arrays, returns either a scalar or a 1D
        array of metric values. Called n_bootstrap times.
    n_bootstrap : int, default 1000
        Number of bootstrap iterations.
    random_state : int, default 0
        Seed for the numpy random generator.

    Returns
    -------
    np.ndarray
        Shape (n_bootstrap,) if metric_fn returns a scalar,
        or (n_bootstrap, m) if metric_fn returns a 1D array of length m.
    """
    rng = np.random.default_rng(random_state)
    true_arr = np.asarray(true_arr)
    pred_arr = np.asarray(pred_arr)

    if len(true_arr) != len(pred_arr):
        raise ValueError(
            f"true_arr and pred_arr have different lengths: "
            f"{len(true_arr)} vs {len(pred_arr)}."
        )

    # Index sample positions by their true-label class. Stratified
    # resampling draws from these per-class index pools separately.
    unique_classes = np.unique(true_arr)
    class_indices = [np.where(true_arr == c)[0] for c in unique_classes]

    bootstrap_metrics = []
    for _ in range(n_bootstrap):
        # Resample within each class; concatenate to recover a full sample.
        resampled_idx = np.concatenate([
            rng.choice(idx, size=len(idx), replace=True)
            for idx in class_indices
        ])
        boot_true = true_arr[resampled_idx]
        boot_pred = pred_arr[resampled_idx]
        bootstrap_metrics.append(metric_fn(boot_true, boot_pred))

    return np.array(bootstrap_metrics)


print("stratified_bootstrap is defined.")
print(f"  Default n_bootstrap: 1000")
print(f"  Default random_state: 0")
print("  Returns a numpy array — 1D for scalar metrics, 2D for vector metrics.")

In [ ]:
# Practice 4.1 — your code here.
#
# Goal: implement compute_null_comparison. Branch on the precommit;
# on each branch, compute the metric point estimate, run a stratified
# bootstrap for the 95% CI, and return the result dict.

def compute_null_comparison(predicted_labels, true_labels, precommit, classifier_classes,
                             n_bootstrap=1000, random_state=0):
    """
    Compute the null comparison for the in-distribution Wang predictions.

    Branches on precommit["above_chance_null"]["decision"]. Returns a
    result dict whose value shapes depend on the branch (see the section
    intro markdown for the two return shapes).

    Parameters
    ----------
    predicted_labels : array-like
        Classifier predictions for the in-distribution Wang samples.
    true_labels : array-like
        Ground-truth Wang labels (from integrated_matrix's stress_label
        column), same length as predicted_labels.
    precommit : dict
        Notebook 00's precommit dict. The "above_chance_null" key drives
        the branch.
    classifier_classes : array-like
        The classifier's full label space (classifier.classes_). Used
        on the multiclass branch to set the null value at 1/N.
    n_bootstrap : int, default 1000
        Bootstrap iterations. Passed through to stratified_bootstrap.
    random_state : int, default 0
        Bootstrap seed. Passed through to stratified_bootstrap.

    Returns
    -------
    dict
        Five keys: "metric_name", "null_value", "point_estimate",
        "ci_lower", "ci_upper". On the binary path the last three are
        scalars; on the multiclass path they are dicts keyed by Wang
        label. See the section intro for the full shape spec.
    """
    decision = precommit["above_chance_null"]["decision"]

    if decision == "binary_cold_vs_control":
        # The classifier was trained multi-class, but on this branch we
        # evaluate cold-vs-not-cold: a sample is "cold" if its label is
        # "cold_stress" and "not-cold" otherwise. Apply this collapse to
        # BOTH true and predicted labels before computing balanced
        # accuracy.
        #
        # Step 1: Define a local metric function `binary_metric(t, p)`
        #         that:
        #         - Takes (t, p) arrays of raw multi-class labels.
        #         - Collapses both to boolean "is cold_stress" arrays.
        #         - Returns balanced_accuracy_score on the boolean
        #           arrays.
        #         Hint: balanced_accuracy_score accepts boolean arrays
        #               directly — no need to cast to int.
        #
        # Step 2: Compute the point estimate by calling binary_metric on
        #         the full (un-resampled) true_labels and predicted_labels.
        #
        # Step 3: Run stratified_bootstrap with binary_metric. Pass
        #         n_bootstrap and random_state through. The return is
        #         a 1D array of length n_bootstrap.
        #
        # Step 4: Compute ci_lower and ci_upper from the 2.5 and 97.5
        #         percentiles of the bootstrap array.
        #         Hint: np.percentile(arr, [2.5, 97.5]) returns both at
        #               once.
        #
        # Step 5: Return the result dict with these keys:
        #             "metric_name":     "balanced_accuracy"
        #             "null_value":      0.50
        #             "point_estimate":  your scalar point estimate
        #             "ci_lower":        your scalar 2.5 percentile
        #             "ci_upper":        your scalar 97.5 percentile

        raise NotImplementedError(
            "Practice 4.1 — implement the binary cold-vs-control path."
        )

    elif decision == "multiclass_behavior":
        # The classifier was trained multi-class; this branch keeps the
        # full prediction distribution and asks whether cold samples get
        # called "cold" specifically (per-class recall), against a 1/N
        # uniform-chance null where N is the number of classifier classes.
        #
        # Compute per-class recall on the two Wang labels — "control"
        # and "cold_stress" — and bootstrap each.
        #
        # Step 1: Define a local metric function `multiclass_metric(t, p)`
        #         that:
        #         - Takes (t, p) arrays of multi-class labels.
        #         - Returns a 2-element array: [recall_control,
        #           recall_cold_stress].
        #         Hint: recall_score(t, p, labels=WANG_LABELS,
        #                            average=None, zero_division=0)
        #               returns the per-label recall array in the order
        #               matching the labels= argument.
        #
        # Step 2: Compute the point estimate by calling multiclass_metric
        #         on the full (un-resampled) labels. Unpack the 2-element
        #         result into a dict keyed by class:
        #             {"control": ..., "cold_stress": ...}
        #
        # Step 3: Run stratified_bootstrap with multiclass_metric. The
        #         returned array has shape (n_bootstrap, 2).
        #
        # Step 4: Compute per-class 2.5 and 97.5 percentiles. axis=0
        #         on np.percentile reduces over the bootstrap iterations,
        #         leaving a 2-element array. Pack each into a dict
        #         keyed by class, matching point_estimate's shape.
        #
        # Step 5: Compute the null value: 1.0 / len(classifier_classes).
        #
        # Step 6: Return the result dict with these keys:
        #             "metric_name":     "per_class_recall"
        #             "null_value":      your 1/N scalar
        #             "point_estimate":  {"control": ..., "cold_stress": ...}
        #             "ci_lower":        {"control": ..., "cold_stress": ...}
        #             "ci_upper":        {"control": ..., "cold_stress": ...}

        raise NotImplementedError(
            "Practice 4.1 — implement the multiclass behavior path."
        )

    else:
        # Section 1 already validated this; defensive belt-and-suspenders.
        raise ValueError(f"Unexpected precommit decision: {decision!r}")


# Sanity prints — uncomment after both branches are written.
# result = compute_null_comparison(
#     predicted_labels=predictions["in_distribution"]["predicted_label"],
#     true_labels=predictions["in_distribution"]["true_stress_label"],
#     precommit=precommit,
#     classifier_classes=classifier_bundle["classifier"].classes_,
# )
# print(f"Branch ran: {result['metric_name']!r}")
# print(f"Point estimate: {result['point_estimate']}")
# print(f"CI lower:       {result['ci_lower']}")
# print(f"CI upper:       {result['ci_upper']}")

In [ ]:
# ----- Call the function -----
in_part = predictions["in_distribution"]

null_result = compute_null_comparison(
    predicted_labels=in_part["predicted_label"],
    true_labels=in_part["true_stress_label"],
    precommit=precommit,
    classifier_classes=classifier_bundle["classifier"].classes_,
)


# ----- Validate the return shape -----
# If the student's function returned the wrong shape, the display block
# below would produce confusing output. These explicit checks point at
# the actual problem.

decision = precommit["above_chance_null"]["decision"]

REQUIRED_KEYS = {"metric_name", "null_value", "point_estimate", "ci_lower", "ci_upper"}

if not isinstance(null_result, dict):
    raise TypeError(
        f"compute_null_comparison returned {type(null_result).__name__}, "
        "but a dict is expected. See the docstring's Returns block."
    )

missing_keys = REQUIRED_KEYS - set(null_result.keys())
if missing_keys:
    raise KeyError(
        f"compute_null_comparison's return is missing key(s): {sorted(missing_keys)}. "
        f"Expected exactly {sorted(REQUIRED_KEYS)}."
    )

if decision == "binary_cold_vs_control":
    expected_metric_name = "balanced_accuracy"
    expected_null_value = 0.50
    for key in ("point_estimate", "ci_lower", "ci_upper"):
        if not isinstance(null_result[key], (float, int, np.floating, np.integer)):
            raise TypeError(
                f"On the binary path, null_result[{key!r}] should be a scalar; "
                f"got {type(null_result[key]).__name__}."
            )

elif decision == "multiclass_behavior":
    expected_metric_name = "per_class_recall"
    expected_null_value = 1.0 / len(classifier_bundle["classifier"].classes_)
    for key in ("point_estimate", "ci_lower", "ci_upper"):
        if not isinstance(null_result[key], dict):
            raise TypeError(
                f"On the multiclass path, null_result[{key!r}] should be a dict "
                f"keyed by Wang label; got {type(null_result[key]).__name__}."
            )
        if set(null_result[key].keys()) != set(WANG_LABELS):
            raise KeyError(
                f"null_result[{key!r}] should have keys exactly {sorted(WANG_LABELS)}; "
                f"got {sorted(null_result[key].keys())}."
            )

if null_result["metric_name"] != expected_metric_name:
    raise ValueError(
        f"null_result['metric_name'] = {null_result['metric_name']!r}, "
        f"but expected {expected_metric_name!r} for the {decision!r} branch."
    )
if not np.isclose(null_result["null_value"], expected_null_value):
    raise ValueError(
        f"null_result['null_value'] = {null_result['null_value']}, "
        f"but expected {expected_null_value} for the {decision!r} branch."
    )


# ----- Add bootstrap-config audit fields -----
# The function computed metrics; the audit fields (n_bootstrap, seed)
# are the responsibility of the calling code so the function's return
# stays focused on the metrics themselves. Section 6 reads these into
# the JSON's audit block.
null_result["n_bootstrap"]    = 1000
null_result["bootstrap_seed"] = 0


# ----- Display the comparison -----
print(f"Null comparison branch: {decision!r}")
print(f"Metric: {null_result['metric_name']}")
print(f"Null value (chance baseline): {null_result['null_value']:.4f}")
print(f"Bootstrap: {null_result['n_bootstrap']} iterations, seed={null_result['bootstrap_seed']}")
print()

if decision == "binary_cold_vs_control":
    pe = null_result["point_estimate"]
    lo = null_result["ci_lower"]
    hi = null_result["ci_upper"]
    null_val = null_result["null_value"]

    print(f"  Point estimate:           {pe:.4f}")
    print(f"  95% CI:                   [{lo:.4f}, {hi:.4f}]")
    print(f"  Above null (point):       {pe > null_val}")
    print(f"  Above null (CI lower):    {lo > null_val}")

elif decision == "multiclass_behavior":
    null_val = null_result["null_value"]
    for label in WANG_LABELS:
        pe = null_result["point_estimate"][label]
        lo = null_result["ci_lower"][label]
        hi = null_result["ci_upper"][label]

        print(f"  {label}")
        print(f"    Point estimate:         {pe:.4f}")
        print(f"    95% CI:                 [{lo:.4f}, {hi:.4f}]")
        print(f"    Above null (point):     {pe > null_val}")
        print(f"    Above null (CI lower):  {lo > null_val}")

**What just happened:**

- Your `compute_null_comparison` function ran on the in-distribution Wang predictions, branched on `precommit["above_chance_null"]["decision"]`, computed the point estimate of the chosen metric, and ran a 1000-iteration stratified bootstrap for the 95% CI.
- The result was validated against the contract for the active branch and the bootstrap-config audit fields (`n_bootstrap`, `bootstrap_seed`) were attached. `null_result` is now the canonical dict for what gets folded into `transfer_summary.json` by Section 6.

**Bootstrap reproducibility.** The fixed seed (`random_state=0`) means re-running this section on the same predictions produces identical CI bounds. If you re-run Notebook 02 with a different `C_VALUE` and your classifier produces different predictions, the CI will change — but for a given predictions array, the CI is deterministic. This is part of why Section 2 wrote `transfer_predictions.parquet` to disk: if a reviewer asks "what bootstrap CI did you report?", you can reload that parquet and re-run this section to reproduce the number exactly.

**Why stratified.** Wang has 3 control and 9 cold samples in-distribution — a 25/75 imbalance. A naive bootstrap could occasionally produce a resample with zero controls or zero colds purely by chance. On a no-control resample, "control" recall is undefined; on a no-cold resample, balanced accuracy is reduced to the cold-class half only. Stratifying — resampling within each class separately at fixed per-class sample sizes — keeps every iteration meaningful. The cost is one extra line of code per resample (`np.concatenate([...])`); the benefit is interpretable CIs at the small sample sizes Wang has.

**A note on the multiclass null.** On the multiclass branch, the null value is `1/N` where N is the classifier's full class count (8 or 9). This is the uniform-chance baseline: if the classifier emitted predictions uniformly at random across its N known classes, the expected recall for any one class would be `1/N`. A "control" recall above `1/N` means the classifier puts control samples in the "control" bin more often than a random emitter would. The null is *not* `1/2` even though Wang carries only 2 labels — because the classifier doesn't know Wang carries only 2 labels; it has N options to pick from for each sample.

**In memory now:**

- `null_result` — the comparison dict, with seven keys after the consumer attached the audit fields. Section 6 reads this directly.
- `stratified_bootstrap` and `compute_null_comparison` — both functions still callable. If you want to re-run with a different bootstrap iteration count for diagnostic purposes (e.g., a 100-iteration run to spot-check before the full 1000), the function signature accepts `n_bootstrap` as a parameter.

**Forward pointer.** Section 5 builds the within-platform vs transfer comparison artifact — per-class recall and precision from `classifier_metrics` (NB02) joined against per-class recall and precision from `transfer_metrics_in` (Section 3), with deltas. Section 6 is the Pattern D gate, which uses `null_result` for its Part 2 check (point estimate vs null, CI lower bound vs null) and `transfer_metrics_in` vs `classifier_metrics` recall for its Part 3 check.

### 5) Comparison vs within-platform

Notebook 02 measured the classifier's performance on a held-out AtGenExpress test split — *within-platform*, same dataset as training. Section 3 measured the classifier's performance on the in-distribution Wang samples — *transfer*, different dataset. Section 5 puts the two numbers next to each other for the two classes Wang has in common with the classifier's training space.

**What gets compared.** Per-class recall and per-class precision for `control` and `cold_stress`, plus the supports. The comparison produces a two-row DataFrame (one row per Wang label) with nine columns: the four point metrics (within / transfer × recall / precision), the two deltas (transfer minus within, for recall and precision), and the two supports (`n_within_platform`, `n_transfer`).

**What does not get compared.** Confusion matrices. Notebook 02's `classifier_confusion.parquet` is square over the classifier's full 8- or 9-class space; Section 3's `transfer_confusion.parquet` is rectangular (2 rows of true Wang labels × N columns of classifier classes). A direct confusion-vs-confusion comparison would only be meaningful in the 2×2 corner where the row and column labels match `{"control", "cold_stress"}` — and at that corner, the comparison is just the per-class precision and recall numbers already in this artifact. The per-class table is the meaningful comparison; the side-by-side confusion display would mostly be empty.

**How the deltas are signed.** `recall_delta = transfer_recall - within_platform_recall`, same for precision. A negative delta means the classifier does *worse* on Wang than on AtGenExpress — the typical outcome for cross-platform transfer. A positive delta means transfer is *better* than within-platform, which is unusual and worth investigating: it could mean integration worked exceptionally well, or it could mean the within-platform numbers were underestimated (small test split, unlucky class imbalance), or — most concerning — that there's some form of dataset leakage that's making transfer artificially easy.

**How the gate uses this.** Section 6's Pattern D gate has three parts; Part 3 is a "sanity vs within-platform" check that reads the `recall_delta` column from this artifact. A delta within +0.10 of within-platform passes; a delta of +0.10 to +0.20 is partial; a delta exceeding +0.20 fails. The check is one-sided: very negative deltas (transfer much worse than within) pass Part 3 — they're a *substantive* problem the gate's Part 2 (above-null) catches, not a *suspicious-numbers* problem Part 3 is looking for.

In [ ]:
# ----- Helper: filter a metrics DataFrame to Wang's two labels -----
# Both classifier_metrics (from Notebook 02, in memory after Section 1)
# and transfer_metrics_in (from Section 3) follow the same row schema:
# per-class rows in a 'class' column plus 'macro_avg' and 'weighted_avg'
# average rows. We want just the two Wang-label rows, in fixed order.

def _filter_to_wang_labels(metrics_df):
    """
    Return a DataFrame indexed by class, with rows for WANG_LABELS in
    fixed order, carrying only the recall/precision/support columns.
    """
    return (
        metrics_df
        .set_index("class")
        .loc[WANG_LABELS, ["recall", "precision", "support"]]
    )


within   = _filter_to_wang_labels(classifier_metrics)
transfer = _filter_to_wang_labels(transfer_metrics_in)


# ----- Build the comparison DataFrame -----
# class as a column with a RangeIndex (matching the convention used by
# transfer_metrics.parquet and classifier_metrics.parquet). Column order
# follows the scoping handoff's schema: identifier, then both within
# metrics, then both transfer metrics, then both deltas, then both
# supports.

comparison = pd.DataFrame({
    "class":                     WANG_LABELS,
    "within_platform_recall":    within["recall"].to_numpy(),
    "within_platform_precision": within["precision"].to_numpy(),
    "transfer_recall":           transfer["recall"].to_numpy(),
    "transfer_precision":        transfer["precision"].to_numpy(),
    "recall_delta":              (transfer["recall"]    - within["recall"]).to_numpy(),
    "precision_delta":           (transfer["precision"] - within["precision"]).to_numpy(),
    "n_within_platform":         within["support"].astype(int).to_numpy(),
    "n_transfer":                transfer["support"].astype(int).to_numpy(),
})


# ----- Display -----
# Floats to three decimals; ints display naturally. Supports go last in
# the table for a reader scanning left-to-right: the metrics and their
# deltas first, the sample-size context at the right.

print("Within-platform (NB02) vs transfer (NB03), per Wang label:")
print()
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.3f}"))


# ----- Write -----
comparison_path = OUTPUT_DIR / "transfer_vs_within_comparison.parquet"
comparison.to_parquet(comparison_path)


# ----- Sanity-load -----
reloaded = pd.read_parquet(comparison_path)
if len(reloaded) != len(comparison):
    raise IOError(
        f"transfer_vs_within_comparison.parquet round-trip failed: "
        f"wrote {len(comparison)} rows, read back {len(reloaded)}."
    )
if list(reloaded.columns) != list(comparison.columns):
    raise IOError(
        f"transfer_vs_within_comparison.parquet round-trip failed: "
        f"column order mismatch.\n"
        f"  wrote:     {list(comparison.columns)}\n"
        f"  read back: {list(reloaded.columns)}"
    )

print()
print(f"Written: {comparison_path}")

**What just happened:**

- Notebook 02's per-class recall and precision (the within-platform numbers, on AtGenExpress's held-out test split) were joined against Section 3's per-class recall and precision (the transfer numbers, on Wang's in-distribution samples) for the two Wang labels.
- Deltas were computed in the transfer-minus-within direction. The comparison was written to `transfer_vs_within_comparison.parquet`.

**Reading the table.** Two rows, one per Wang label. For each row, the four metric columns (`within_platform_recall`, `within_platform_precision`, `transfer_recall`, `transfer_precision`) describe the same scalar from two different vantage points — same classifier, same class, different evaluation set. The two delta columns make the transfer-vs-within gap visible directly. The two support columns provide the sample-size context: a 0.20 recall_delta means something different at `n_transfer=3` (one extra correctly-classified sample) than at `n_transfer=30` (six extra). For Wang in-distribution, expect `n_transfer` around 3 for `control` and 9 for `cold_stress`.

**Sign conventions and what to expect.**

- *Typical pattern:* negative deltas. Cross-platform transfer usually loses some performance — there's no platform-specific quirk the classifier learned during training that's also helping it on Wang. A `recall_delta` of -0.05 to -0.20 is normal.
- *Suspicious pattern:* large positive deltas. Transfer outperforming within-platform by more than ~0.10 is rare and worth checking — the within-platform numbers might be underestimated (small test split, unlucky class imbalance), the transfer numbers might be inflated (small Wang sample size means high variance), or — least likely but most consequential — there's some form of dataset leakage between AtGenExpress and Wang that's making the transfer evaluation artificially easy.
- *Substantively bad pattern:* very negative deltas. A `recall_delta` near -1.0 means the classifier is completely missing Wang samples of that class. Section 4's null comparison catches this as a "below chance" finding; Section 6's gate Part 2 then issues the fail verdict.

**In memory now:**

- `comparison` — the assembled DataFrame, also on disk at `OUTPUT_DIR / "transfer_vs_within_comparison.parquet"`. Section 6 reads it (specifically the `recall_delta` column) for the gate's Part 3 check.
- `within`, `transfer` — the two filtered intermediate DataFrames. Not referenced after this cell; available if you want to inspect them separately during debugging.

**Forward pointer.** Section 6 is the Pattern D gate that reads everything Sections 1 through 5 produced — the `null_result` dict from Section 4, the per-class metrics from Section 3, and this comparison artifact — and renders a verdict on whether the transfer evaluation is trustworthy enough to write up. The gate's three parts: sample-size sufficiency (Part 1), above-null point estimate (Part 2, from `null_result`), and sanity-vs-within-platform (Part 3, from the `recall_delta` column of this artifact).

### 6) Gate: is the transfer evaluation trustworthy?


**What this gate is asking.** The earlier gates in the chain (Notebook 01's integration gate, Notebook 02's accuracy gate) asked the same question: *should you trust the next notebook's input?* This gate asks a different question: *should you trust the answer you are about to write up in your Week 4 paper?* Same stop-and-judge mechanic, but at the highest stakes in the chain — the paper's headline claim depends on whether the transfer evaluation came out trustworthy.

**Three parts.** Three independent checks, composed worst-of-three for the overall verdict:

1. **Sample size sufficiency.** Is `n_eval` large enough that the per-class numbers carry meaning?
2. **Above-null point estimate.** Does the transfer-evaluation metric beat the chance null you committed to in Notebook 00? Section 4's `null_result` carries the inputs.
3. **Sanity vs within-platform.** Does transfer-evaluation come in *suspiciously high* relative to within-platform performance? This is a one-sided check — very *low* transfer relative to within-platform is a substantive finding (Part 2 catches it), not a methodological flag.

The next cell lays out pass / partial / fail criteria for each part. The cell after that auto-computes the diagnostic values you need, then asks you to fill in your verdict and rationale for each part. The cell composes the overall verdict, refuses to write `transfer_summary.json` if the overall is `fail`, and writes the file otherwise.

**On the rationales.** Unlike the earlier gates, the rationales you write here become draft material for the paper's limitations and methodology paragraphs. Write them the way you would for a peer reviewer — defend the verdict using the diagnostic values, name the trade-offs you are carrying forward, and own the uncertainties honestly. A rationale that just restates the verdict ("pass because n_eval is large enough") is not useful; a rationale that contextualises the verdict ("pass because n_eval = 12 supports 3-control / 9-cold per-class recall estimates with the small but interpretable per-class sample sizes Wang's experimental design produces") is.

**Part 1 — Sample size sufficiency.**

The transfer evaluation runs on Wang's in-distribution samples (`n_eval`). Wang's design produces 3 control and 9 cold samples in-distribution (per the Wang Design Finding Memo), so `n_eval` should typically be 12.

| Verdict | Criterion |
|---|---|
| pass | `n_eval` ≥ 9 |
| partial | `n_eval` ∈ {6, 7, 8} |
| fail | `n_eval` < 6 |

A pass here does not mean Wang has a lot of samples — 12 is small by any conventional standard. It means the sample size supports the per-class metrics you reported. Three samples per class is the rough floor for meaningful recall numbers; below six total samples (`n_eval` < 6) the per-class numbers stop being interpretable.

---

**Part 2 — Above-null point estimate.**

Did the classifier's transfer-evaluation metric beat the chance null you committed to in Notebook 00?

On both precommit branches, the per-class check uses the same three-tier criterion:

| Verdict | Criterion |
|---|---|
| pass | point ≥ null + 0.05 AND CI lower > null |
| partial | point > null, but either margin < 0.05 OR CI lower ≤ null |
| fail | point ≤ null |

- **Binary path** (`metric_name == "balanced_accuracy"`): apply the criterion to the scalar point estimate against the 0.50 null.
- **Multiclass path** (`metric_name == "per_class_recall"`): apply the criterion to each Wang label's recall against the `1/N` null. Compose: the worst per-class verdict is the part's verdict (pass-pass → pass; pass-partial → partial; pass-fail → fail; partial-fail → fail; fail-fail → fail).

The `+ 0.05` buffer keeps the pass criterion meaningfully above chance — a balanced accuracy of 0.51 is technically above 0.50, but at Wang's small sample sizes it is well within sampling noise. The CI-lower requirement is what makes the pass an *interval* claim rather than just a *point* claim: the bootstrap distribution has to land mostly above null, not just its centre.

---

**Part 3 — Sanity vs within-platform.**

This part reads `recall_delta` from the comparison artifact (Section 5). The delta is signed `transfer − within`: cross-platform transfer typically loses some performance, so most `recall_delta` values are negative — and negative deltas pass this part automatically. The concern is the *suspiciously positive* case.

| Verdict | Criterion |
|---|---|
| pass | max(`recall_delta` across the two rows) ≤ +0.10 |
| partial | max(`recall_delta`) ∈ (+0.10, +0.20] |
| fail | max(`recall_delta`) > +0.20 |

A fail here does not necessarily mean the transfer evaluation is wrong — it means the result is surprising enough that you need to rule out leakage, dataset overlap, or some other methodological problem before writing it up. The fail criterion exists because in the literature, "transfer accuracy meaningfully exceeds within-platform accuracy" is almost always a sign of a bug or data leak, not a true generalization story.

---

**Overall.** The overall verdict is the worst of the three part verdicts (worse-than: `fail` > `partial` > `pass`). On `fail`, the cell raises `RuntimeError` and refuses to write `transfer_summary.json` — the paper's transfer claim cannot be written up with the current numbers. On `pass` or `partial`, the file is written and the rationales become draft material for the paper.

In [ ]:
# ============================================================
# Auto-computed diagnostic values
# ============================================================

decision = precommit["above_chance_null"]["decision"]

# Part 1: Sample size sufficiency
n_eval = len(predictions["in_distribution"]["sample_id"])
n_ood  = len(predictions["out_of_distribution"]["sample_id"])

# Part 2: Above-null point estimate
null_value = null_result["null_value"]

if decision == "binary_cold_vs_control":
    margin_above_null     = null_result["point_estimate"] - null_value
    above_null            = null_result["point_estimate"] > null_value
    above_null_lower_bound = null_result["ci_lower"]      > null_value

elif decision == "multiclass_behavior":
    margin_above_null = {
        label: null_result["point_estimate"][label] - null_value
        for label in WANG_LABELS
    }
    above_null = {
        label: null_result["point_estimate"][label] > null_value
        for label in WANG_LABELS
    }
    above_null_lower_bound = {
        label: null_result["ci_lower"][label] > null_value
        for label in WANG_LABELS
    }

# Part 3: Sanity vs within-platform
max_recall_delta = float(comparison["recall_delta"].max())


# ============================================================
# Display diagnostics — use these to inform your verdicts below.
# ============================================================

print("=" * 64)
print("Gate diagnostics (auto-computed):")
print("=" * 64)
print()

print("Part 1 — Sample size sufficiency")
print(f"  n_eval (in-distribution Wang samples): {n_eval}")
print(f"  Thresholds: pass ≥ 9, partial ∈ {{6,7,8}}, fail < 6")
print()

print("Part 2 — Above-null point estimate")
print(f"  Metric:     {null_result['metric_name']}")
print(f"  Null value: {null_value:.4f}")
if decision == "binary_cold_vs_control":
    print(f"  Point estimate:        {null_result['point_estimate']:.4f}")
    print(f"  95% CI:                [{null_result['ci_lower']:.4f}, {null_result['ci_upper']:.4f}]")
    print(f"  Margin above null:     {margin_above_null:+.4f}")
    print(f"  Above null (point):    {above_null}")
    print(f"  Above null (CI lower): {above_null_lower_bound}")
elif decision == "multiclass_behavior":
    for label in WANG_LABELS:
        pe = null_result["point_estimate"][label]
        lo = null_result["ci_lower"][label]
        hi = null_result["ci_upper"][label]
        print(f"  {label}:")
        print(f"    Point estimate:        {pe:.4f}")
        print(f"    95% CI:                [{lo:.4f}, {hi:.4f}]")
        print(f"    Margin above null:     {margin_above_null[label]:+.4f}")
        print(f"    Above null (point):    {above_null[label]}")
        print(f"    Above null (CI lower): {above_null_lower_bound[label]}")
print(f"  Thresholds: pass = margin ≥ 0.05 AND CI lower > null;")
print(f"              partial = above null but margin < 0.05 OR CI lower ≤ null;")
print(f"              fail = at or below null.")
if decision == "multiclass_behavior":
    print(f"              (Multiclass: compose worst per-class verdict.)")
print()

print("Part 3 — Sanity vs within-platform")
for _, row in comparison.iterrows():
    print(f"  {row['class']}: recall_delta = {row['recall_delta']:+.4f}")
print(f"  max recall_delta: {max_recall_delta:+.4f}")
print(f"  Thresholds: pass ≤ +0.10, partial ∈ (+0.10, +0.20], fail > +0.20")
print()
print("=" * 64)
print()


# ============================================================
# Practice — your verdicts and rationales.
# ============================================================
#
# For each part below, replace "PLACEHOLDER" with one of "pass",
# "partial", or "fail" based on the diagnostic values above and
# the criteria from the previous markdown cell. Replace the
# rationale strings with a one-or-two-sentence defense in your
# own words. The validation step below enforces non-placeholder,
# minimum-length rationales.
#
# The "overall" entry's verdict is computed below — leave its
# verdict as None and fill in only its rationale. The overall
# rationale becomes draft material for your Week 4 paper's
# limitations paragraph.

gate = {
    "part_1_sample_size": {
        "verdict":   "PLACEHOLDER",
        "rationale": "PLACEHOLDER — defend your Part 1 verdict here.",
    },
    "part_2_above_null": {
        "verdict":   "PLACEHOLDER",
        "rationale": "PLACEHOLDER — defend your Part 2 verdict here.",
    },
    "part_3_sanity_vs_within_platform": {
        "verdict":   "PLACEHOLDER",
        "rationale": "PLACEHOLDER — defend your Part 3 verdict here.",
    },
    "overall": {
        "verdict":   None,  # computed below — leave as None.
        "rationale": "PLACEHOLDER — defend the rollup judgment as a "
                     "limitations-paragraph draft for the Week 4 paper.",
    },
}


# ============================================================
# Structural validation
# ============================================================

ALLOWED_VERDICTS    = {"pass", "partial", "fail"}
PLACEHOLDER_MARKER  = "PLACEHOLDER"
MIN_RATIONALE_CHARS = 30

per_part_names = [
    "part_1_sample_size",
    "part_2_above_null",
    "part_3_sanity_vs_within_platform",
]

for name in per_part_names:
    entry = gate[name]

    if entry["verdict"] not in ALLOWED_VERDICTS:
        raise ValueError(
            f"gate['{name}']['verdict'] = {entry['verdict']!r}. "
            f"Replace with one of {sorted(ALLOWED_VERDICTS)} based on the "
            f"diagnostic values printed above and the criteria in the "
            f"previous markdown cell."
        )

    rationale = entry["rationale"].strip()
    if PLACEHOLDER_MARKER in rationale:
        raise ValueError(
            f"gate['{name}']['rationale'] still contains the placeholder "
            "text. Replace it with your own defense of the verdict."
        )
    if len(rationale) < MIN_RATIONALE_CHARS:
        raise ValueError(
            f"gate['{name}']['rationale'] is too short ({len(rationale)} chars). "
            f"Write at least one sentence (~{MIN_RATIONALE_CHARS} characters)."
        )


# ============================================================
# Compose the overall verdict — worst of the three parts.
# ============================================================

per_part_verdicts = [gate[name]["verdict"] for name in per_part_names]
if "fail" in per_part_verdicts:
    overall_verdict = "fail"
elif "partial" in per_part_verdicts:
    overall_verdict = "partial"
else:
    overall_verdict = "pass"

gate["overall"]["verdict"] = overall_verdict

# Validate the overall rationale.
overall_rationale = gate["overall"]["rationale"].strip()
if PLACEHOLDER_MARKER in overall_rationale:
    raise ValueError(
        "gate['overall']['rationale'] still contains the placeholder text. "
        "Replace it with your defense of the rollup judgment — this rationale "
        "becomes draft material for your Week 4 paper's limitations paragraph."
    )
if len(overall_rationale) < MIN_RATIONALE_CHARS:
    raise ValueError(
        f"gate['overall']['rationale'] is too short ({len(overall_rationale)} chars). "
        f"Write at least one sentence (~{MIN_RATIONALE_CHARS} characters)."
    )


# ============================================================
# Refuse on fail — no file write.
# ============================================================

if overall_verdict == "fail":
    raise RuntimeError(
        "Overall gate verdict is 'fail'. transfer_summary.json will not be "
        "written. The transfer claim cannot be written up with the current "
        "numbers — return to the failing part, re-examine the upstream "
        "evaluation, and decide whether the transfer evaluation can be "
        "salvaged. If not, the paper's headline finding becomes the "
        "negative result itself, not a transfer claim."
    )


# ============================================================
# Assemble transfer_summary.json payload.
# ============================================================

import json
from datetime import datetime, timezone


def _coerce_payload(value, is_multiclass):
    """
    Cast a payload value to JSON-friendly primitives. On the binary
    path, value is a scalar — coerce to float. On the multiclass path,
    value is a dict keyed by class label — coerce each value to float.
    """
    if is_multiclass:
        return {label: float(value[label]) for label in WANG_LABELS}
    else:
        return float(value)


is_multiclass = (decision == "multiclass_behavior")

# OOD metrics: structured copy of Section 3's transfer_metrics_ood as a
# JSON-friendly list, or None on the 'exclude' precommit path.
if transfer_metrics_ood is None:
    ood_metrics_payload = None
else:
    ood_metrics_payload = {
        "per_class": [
            {
                "class":     str(row["class"]),
                "precision": float(row["precision"]),
                "recall":    float(row["recall"]),
                "f1":        float(row["f1"]),
                "support":   int(row["support"]),
            }
            for _, row in transfer_metrics_ood.iterrows()
        ]
    }

summary = {
    "metric_path":             decision,
    "metric_name":             null_result["metric_name"],
    "null_value":              float(null_value),
    "point_estimate":          _coerce_payload(null_result["point_estimate"], is_multiclass),
    "ci_lower":                _coerce_payload(null_result["ci_lower"],       is_multiclass),
    "ci_upper":                _coerce_payload(null_result["ci_upper"],       is_multiclass),
    "n_bootstrap":             int(null_result["n_bootstrap"]),
    "bootstrap_seed":          int(null_result["bootstrap_seed"]),
    "above_null":              (above_null              if is_multiclass else bool(above_null)),
    "above_null_lower_bound":  (above_null_lower_bound  if is_multiclass else bool(above_null_lower_bound)),
    "n_eval":                  int(n_eval),
    "n_ood":                   int(n_ood),
    "ood_metrics":             ood_metrics_payload,
    "gate":                    gate,
    "precommit_summary": {
        "data_source":         precommit["data_source_and_scope"]["source"],
        "wang_168h_handling":  precommit["wang_168h_handling"]["decision"],
        "above_chance_null":   precommit["above_chance_null"]["decision"],
    },
    "run_metadata": {
        "timestamp":           datetime.now(timezone.utc).isoformat(),
        "notebook":            "03_wang_evaluation.ipynb",
        "random_seeds":        {"bootstrap": int(null_result["bootstrap_seed"])},
    },
}


# ============================================================
# Write and confirm.
# ============================================================

summary_path = OUTPUT_DIR / "transfer_summary.json"
with summary_path.open("w") as f:
    json.dump(summary, f, indent=2)

print(f"Overall gate verdict: {overall_verdict!r}")
print(f"transfer_summary.json written to {summary_path}")

if overall_verdict == "partial":
    print()
    print("=" * 64)
    print("PARTIAL: writing up the Week 4 paper is permitted, but document")
    print("the trade-off you are carrying forward in the paper's")
    print("limitations paragraph. The 'overall' rationale you just wrote")
    print("is the starting point for that paragraph.")
    print("=" * 64)

### 7) Closeout: confirm artifacts on disk; submit the Week 4 paper and final presentation

You have reached the end of the R1-Q4 notebook chain. This section confirms that all five files Notebook 03 produced are on disk, prints a headline summary of the run that you can copy into your paper's abstract, and points you to the Week 4 paper and final presentation deliverables.

**Five files were written across Sections 2–6.** No new files get written here — the closeout just verifies what's on disk:

- **`transfer_predictions.parquet`** (Section 2) — per-sample Wang predictions with sample IDs, predicted labels, prediction probabilities, true labels, and the OOD flag. The sample-level archival record your paper can reference for sample-by-sample verification.
- **`transfer_metrics.parquet`** (Section 3) — per-class precision, recall, F1, and support for the in-distribution Wang samples. The results table.
- **`transfer_confusion.parquet`** (Section 3) — the rectangular confusion matrix, two true-label rows by N predicted-label columns. The results figure.
- **`transfer_vs_within_comparison.parquet`** (Section 5) — within-platform (NB02) vs transfer (NB03) per-class numbers with recall and precision deltas. The discussion-section comparison.
- **`transfer_summary.json`** (Section 6) — the gate verdict, all of the metric numbers, the OOD side fields, the precommit summary, and the run metadata. The methods-section reference and the limitations-paragraph starting point.

If any file is missing, the next cell raises with a pointer to the section that should have written it. Re-run that section, then re-run this one.

In [ ]:
# ============================================================
# Confirm all five deliverables are on disk
# ============================================================

expected_files = [
    ("transfer_predictions.parquet",          "Section 2"),
    ("transfer_metrics.parquet",              "Section 3"),
    ("transfer_confusion.parquet",            "Section 3"),
    ("transfer_vs_within_comparison.parquet", "Section 5"),
    ("transfer_summary.json",                 "Section 6"),
]

print("Notebook 03 deliverables on disk:")
print()
for filename, source_section in expected_files:
    path = OUTPUT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"{filename} was expected at {path} but is not present. "
            f"Re-run {source_section}, then re-run this cell."
        )
    size_kb = path.stat().st_size / 1024
    print(f"  {filename:<45s}  {size_kb:>10,.1f} KB    ({source_section})")


# ============================================================
# Headline summary of this run
# ============================================================

print()
print("=" * 64)
print("Headline summary of this run:")
print("=" * 64)
print()

print("Precommit (from Notebook 00):")
print(f"  data_source:        {precommit['data_source_and_scope']['source']!r}")
print(f"  wang_168h_handling: {precommit['wang_168h_handling']['decision']!r}")
print(f"  above_chance_null:  {precommit['above_chance_null']['decision']!r}")
print()

print("Sample sizes:")
print(f"  n_eval (in-distribution Wang): {n_eval}")
print(f"  n_ood:                         {n_ood}")
print()

print(f"Transfer evaluation:")
print(f"  Metric:     {null_result['metric_name']}")
print(f"  Null value: {null_result['null_value']:.4f}")
if decision == "binary_cold_vs_control":
    print(f"  Point estimate: {null_result['point_estimate']:.4f}")
    print(f"  95% CI:         [{null_result['ci_lower']:.4f}, {null_result['ci_upper']:.4f}]")
else:
    for label in WANG_LABELS:
        print(f"  {label}:")
        print(f"    Point estimate: {null_result['point_estimate'][label]:.4f}")
        print(f"    95% CI:         [{null_result['ci_lower'][label]:.4f}, "
              f"{null_result['ci_upper'][label]:.4f}]")
print()

print(f"Gate verdict: {overall_verdict!r}")
print()

print("Primary reference document for your paper writeup:")
print(f"  {OUTPUT_DIR / 'transfer_summary.json'}")


# ============================================================
# End-of-chain marker
# ============================================================

print()
print("=" * 64)
print("R1-Q4 notebook chain complete.")
print("=" * 64)

**Week 4 deliverables.** The R1-Q4 question page in Notion lists the formal Week 4 submission procedure. Two deliverables:

- **The Week 4 paper.** Your write-up of the transfer evaluation. The headline finding goes in the abstract; the methods section cites the precommit, the integration step (Notebook 01), the classifier choice (Notebook 02), and the transfer-evaluation protocol (this notebook). The results section reports the numbers from `transfer_summary.json` and includes the per-class metrics table from `transfer_metrics.parquet`. The discussion contextualises the result against the within-platform comparison (`transfer_vs_within_comparison.parquet`). The limitations paragraph starts from the rationales you wrote in Section 6's gate verdict — particularly the `overall` rationale, which was written specifically for this purpose.

- **The Week 4 final presentation.** Your presentation of the same finding to the cohort. The headline number comes from `transfer_summary.json`; the confusion matrix from `transfer_confusion.parquet` makes a natural slide; the within-platform vs transfer comparison from `transfer_vs_within_comparison.parquet` is the natural "what we expected vs what we found" framing for the methods slide.

**On the rationale-as-draft-material framing.** Section 6 was deliberate about asking you to write the gate rationales the way you would for a peer reviewer — defending the verdict using the diagnostic values, naming the trade-offs, owning the uncertainties. That writing was not bookkeeping. It is the seed text for your paper's limitations paragraph. Open `transfer_summary.json`, copy the `gate` block's rationales into your draft, and edit them into paragraph prose. The mental work of deciding what to write was the part that took judgment; turning the rationale into paragraphs is editorial work that goes much faster from a defended verdict than from a blank page.

**On the precommit.** The precommit dict in `transfer_summary.json["precommit_summary"]` is the methodological audit trail. If a reviewer asks "why did you choose the binary path / the include-as-ood path / the GEO data source," the answer is in Notebook 00's Precommit sections (rationales recorded at the time of choosing) and reflected in this summary file. Cite both — the precommit was made *before* the result was known, which is what makes the result honest.

**The chain.** R1-Q4's notebook chain is now complete: Notebook 00 (orient and precommit), Notebook 01 (integration), Notebook 02 (classifier), Notebook 03 (transfer evaluation — this notebook). Five files have been written to `OUTPUT_DIR` by this notebook alone; combined with NB00's `precommit.json`, NB01's `integrated_matrix.parquet` and `integration_summary.json`, and NB02's `classifier.pkl`, `atgenexpress_test_split.parquet`, `classifier_metrics.parquet`, `classifier_confusion.parquet`, and `classifier_summary.json`, the chain's full output set is in one place — the reproducibility package for the Week 4 submission.